# nb01 Data Pull: DMHC Financial Summary reports

**Purpose**
- Pull the Health Plan Financial Summary report from the DMHC (Department of Managed Health Care) application at `wpso.dmhc.ca.gov/flash/` for the two study plans, and save each response as both a raw HTML snapshot and a parsed CSV.
- Plans, by license number:
  - `933 0355` Local Initiative Health Authority for Los Angeles County, doing business as L.A. Care Health Plan.
  - `933 0426` Health Net Community Solutions, Inc.
- All 13 financial measures plus the Total Enrollees and Medi-Cal Managed Care enrollment counts, annual and quarterly reports.

**Design notes (from the nb00 diagnostics)**
- The Create Report button posts the form to `flash.aspx`; the report is one flat HTML table, one row per plan and reporting period, one column per measure.
- Dates must be formatted as month name plus year (for example `July 2020`); a numeric date crashes the report engine.
- A single 26 year request times out server side, so the pull is chunked into non overlapping windows of at most 5 years. Each plan and window becomes its own CSV, and the union of those files happens later in Tableau, never in Python.

**Prerequisites**
- Python 3 with `requests` and `beautifulsoup4` (already installed for nb00).
- Run from the `tableau_plan_financials/notebooks/` folder.

**Outputs**
- `data/raw/financial_summary_<plan>_<window>.html` and `.csv`, one pair per plan per window.
- Diagnostics in `nb01_data_pull_cell_output.txt`; every pull is wrapped so one failure never stops the others.

In [1]:
# Step 0: mirror all printed output to a text file for easy sharing
import sys
from pathlib import Path

SINK_PATH = Path.cwd() / "nb01_data_pull_cell_output.txt"

_orig_out = getattr(sys, "_nb_orig_stdout", sys.stdout)
_orig_err = getattr(sys, "_nb_orig_stderr", sys.stderr)
sys._nb_orig_stdout, sys._nb_orig_stderr = _orig_out, _orig_err

class _Tee:
    def __init__(self, stream, fh):
        self.stream, self.fh = stream, fh
    def write(self, data):
        self.stream.write(data)
        self.fh.write(data)
        self.fh.flush()
    def flush(self):
        self.stream.flush()
        self.fh.flush()

_sink = open(SINK_PATH, "w")
sys.stdout = _Tee(_orig_out, _sink)
sys.stderr = _Tee(_orig_err, _sink)
print(f"Mirroring cell output to {SINK_PATH.name} (attach this file in the chat)")

Mirroring cell output to nb01_data_pull_cell_output.txt (attach this file in the chat)
Pull plan: 2 plans x 4 windows = 8 reports -> /Users/trinidadcisneros/Documents/Development/trinidadcisneros.github.io/folders/ds_blogs/projects/tableau/tableau_plan_financials/data/raw
Helpers ready.
[financial_summary_lacare_2010_2014] 12.9s, 37,037 bytes, 25 data rows, 21 columns -> financial_summary_lacare_2010_2014.csv
[financial_summary_lacare_2015_2019] 16.3s, 37,408 bytes, 25 data rows, 21 columns -> financial_summary_lacare_2015_2019.csv
[financial_summary_lacare_2020_2023] 13.8s, 31,489 bytes, 20 data rows, 21 columns -> financial_summary_lacare_2020_2023.csv
[financial_summary_lacare_2024_2026] 8.3s, 20,664 bytes, 11 data rows, 21 columns -> financial_summary_lacare_2024_2026.csv
[financial_summary_hncs_2010_2014] 12.4s, 36,149 bytes, 25 data rows, 21 columns -> financial_summary_hncs_2010_2014.csv
[financial_summary_hncs_2015_2019] 14.5s, 36,386 bytes, 25 data rows, 21 columns -> financia

In [2]:
# Step 1: setup, constants, and the pull plan
import csv
import time
import requests
from bs4 import BeautifulSoup

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

FLASH_URL = "https://wpso.dmhc.ca.gov/flash/"
REPORT_URL = "https://wpso.dmhc.ca.gov/flash/flash.aspx"
P = "ctl00$ctl00$MainContent$MainContent$"

PLANS = {
    "lacare": "933 0355",   # Local Initiative Health Authority for Los Angeles County (L.A. Care Health Plan)
    "hncs": "933 0426",     # Health Net Community Solutions, Inc.
}

# Non overlapping windows, at most 5 years each; month name plus year is the only
# date format the report engine accepts. A single 26 year request times out.
WINDOWS = [
    ("2010_2014", "January 2010", "December 2014"),
    ("2015_2019", "January 2015", "December 2019"),
    ("2020_2023", "January 2020", "December 2023"),
    ("2024_2026", "January 2024", "July 2026"),
]

MEASURES = ["tne", "req_tne", "excess", "tne_required", "totalAssets", "totalCurrentAssets",
            "totalCurrentLiabilities", "revenue", "income_loss", "admin_exp", "admin_ratio",
            "med_exp", "medloss_ratio"]
ENROLLMENT = {"0": "Total Enrollees", "1": "Medi-Cal Managed Care"}

S = requests.Session()
S.headers.update({"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X) research pull for a public data blog"})

print(f"Pull plan: {len(PLANS)} plans x {len(WINDOWS)} windows = {len(PLANS) * len(WINDOWS)} reports -> {RAW_DIR}")

In [3]:
# Step 2: helpers, form harvest, report pull, HTML table to CSV
def harvest_hidden_fields():
    """GET the landing page and collect the ASP.NET hidden fields for a fresh POST."""
    r = S.get(FLASH_URL, timeout=60)
    soup = BeautifulSoup(r.text, "html.parser")
    fields = {}
    for inp in soup.find_all("input", type="hidden"):
        if inp.get("name"):
            fields[inp["name"]] = inp.get("value") or ""
    return fields

def build_payload(hidden, plan_value, start, end):
    payload = dict(hidden)
    payload[P + "lbHP"] = plan_value
    payload[P + "lbHPType"] = "0"
    payload[P + "ddlReportType"] = "0"   # Annual and Quarterly
    payload[P + "txtStartDate"] = start
    payload[P + "txtEndDate"] = end
    for i, v in enumerate(MEASURES):
        payload[f"{P}cblFinancial${i}"] = v
    for i, v in ENROLLMENT.items():
        payload[f"{P}cblEnrollment${i}"] = v
    payload[P + "cmdSubmit"] = "Create Report"
    return payload

def parse_report_table(html):
    """Return (headers, rows) from the report's single flat table."""
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    if table is None:
        return [], []
    trs = table.find_all("tr")
    headers = [c.get_text(strip=True) for c in trs[0].find_all(["th", "td"])] if trs else []
    rows = []
    for tr in trs[1:]:
        cells = [c.get_text(strip=True) for c in tr.find_all(["th", "td"])]
        if any(cells):
            rows.append(cells)
    return headers, rows

def pull_report(tag, plan_value, window_tag, start, end):
    """One report pull: save raw HTML and parsed CSV, print diagnostics. Never raises."""
    stem = f"financial_summary_{tag}_{window_tag}"
    try:
        hidden = harvest_hidden_fields()
        payload = build_payload(hidden, plan_value, start, end)
        t0 = time.time()
        r = S.post(REPORT_URL, data=payload, timeout=180)
        elapsed = time.time() - t0
        (RAW_DIR / f"{stem}.html").write_bytes(r.content)
        soup = BeautifulSoup(r.text, "html.parser")
        title = soup.title.get_text(strip=True) if soup.title else "none"
        if "Error" in title:
            print(f"[{stem}] REPORT ERROR after {elapsed:.1f}s ({len(r.content):,} bytes); "
                  f"window may need to be split further")
            return None
        headers, rows = parse_report_table(r.text)
        out_csv = RAW_DIR / f"{stem}.csv"
        with open(out_csv, "w", newline="") as fh:
            w = csv.writer(fh)
            w.writerow(headers)
            w.writerows(rows)
        print(f"[{stem}] {elapsed:.1f}s, {len(r.content):,} bytes, {len(rows)} data rows, "
              f"{len(headers)} columns -> {out_csv.name}")
        return len(rows)
    except Exception as e:
        print(f"[{stem}] FAILED: {type(e).__name__}: {e}")
        return None

print("Helpers ready.")

In [4]:
# Step 3: run every pull, politely spaced
results = {}
for tag, plan_value in PLANS.items():
    for window_tag, start, end in WINDOWS:
        results[(tag, window_tag)] = pull_report(tag, plan_value, window_tag, start, end)
        time.sleep(2)  # be polite to the state server

done = sum(1 for v in results.values() if v is not None)
print(f"\nCompleted {done} of {len(results)} pulls.")

In [5]:
# Step 4: integrity checks across the saved CSVs
# - consistent headers across every file
# - row counts by report type per plan
# - no duplicate (report type, statement date) pairs across a plan's windows
import collections

all_headers = {}
plan_periods = collections.defaultdict(list)
type_counts = collections.defaultdict(collections.Counter)

for tag in PLANS:
    for window_tag, _, _ in WINDOWS:
        f = RAW_DIR / f"financial_summary_{tag}_{window_tag}.csv"
        if not f.exists():
            print(f"missing (pull failed or empty): {f.name}")
            continue
        with open(f, newline="") as fh:
            reader = csv.reader(fh)
            headers = next(reader, [])
            all_headers[f.name] = headers
            for row in reader:
                rec = dict(zip(headers, row))
                key = (rec.get("Report Type", ""), rec.get("Statement Date", ""))
                plan_periods[tag].append(key)
                type_counts[tag][rec.get("Report Type", "")] += 1

header_sets = {tuple(h) for h in all_headers.values()}
print(f"files read: {len(all_headers)}; distinct header layouts: {len(header_sets)}")
if len(header_sets) == 1 and header_sets:
    print("  columns:", list(next(iter(header_sets))))
else:
    for name, h in all_headers.items():
        print(f"  {name}: {len(h)} columns")

for tag in PLANS:
    keys = plan_periods[tag]
    dupes = [k for k, n in collections.Counter(keys).items() if n > 1]
    dates = sorted(k[1] for k in keys if k[1])
    print(f"\n[{tag}] {len(keys)} rows; statement dates {dates[0] if dates else 'none'} "
          f"to {dates[-1] if dates else 'none'}")
    for rtype, n in sorted(type_counts[tag].items()):
        print(f"    {rtype or '(blank)'}: {n} rows")
    if dupes:
        print(f"    DUPLICATE (report type, statement date) pairs across windows: {dupes[:10]}")
    else:
        print("    no duplicate periods across windows")

In [6]:
# Step 5: confirm the output sink
sys.stdout.flush()
print(f"\nAll printed output saved to: {SINK_PATH}")
print(f"File size: {SINK_PATH.stat().st_size:,} bytes")

**Next step**
- Attach `nb01_data_pull_cell_output.txt` in the chat.
- The printed row counts, date ranges, and duplicate check decide the nb02 clean design: numeric parsing, period columns for Tableau, and the annual vs quarterly handling.
- The raw HTML snapshots stay in `data/raw/` so every CSV can be re derived without refetching.